In [12]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import datetime
import csv  # CSV 保存用

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ----------------------------
# Graph Autoencoder (GAE) モデル定義
# ----------------------------
class GraphAutoencoder(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(GraphAutoencoder, self).__init__()
        self.encoder1 = GCNConv(input_dim, hidden_dim)
        self.encoder2 = GATConv(hidden_dim, hidden_dim, heads=4, concat=True)
        self.encoder3 = SAGEConv(hidden_dim * 4, latent_dim)
        self.decoder1 = torch.nn.Linear(latent_dim, hidden_dim)
        self.decoder2 = torch.nn.Linear(hidden_dim, input_dim)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.encoder1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.encoder2(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        z = F.relu(self.encoder3(x, edge_index))
        x_reconstructed = F.relu(self.decoder1(z))
        x_reconstructed = self.decoder2(x_reconstructed)
        return x_reconstructed, z

# ----------------------------
# 欠陥予測用 GNN モデル定義
# ----------------------------
class DefectPredictionGNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(DefectPredictionGNN, self).__init__()
        self.conv1 = GATConv(input_dim, hidden_dim, heads=4, concat=True)
        self.conv2 = GATConv(hidden_dim * 4, hidden_dim, heads=1, concat=False)
        self.fc = torch.nn.Linear(hidden_dim, num_classes)
        self.apply(self._initialize_weights)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.fc(x)
        return x

    def _initialize_weights(self, layer):
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

# ----------------------------
# 異常検知と欠陥予測のステップ
# ----------------------------
def anomaly_detection(data_loader, autoencoder, device):
    autoencoder.eval()
    reconstruction_errors = []
    with torch.no_grad():
        for data in data_loader:
            data = data.to(device)
            x_reconstructed, _ = autoencoder(data)
            loss = F.mse_loss(x_reconstructed, data.x, reduction='none').mean(dim=1)
            reconstruction_errors.append(loss.cpu().numpy())
    return np.concatenate(reconstruction_errors)

def label_defects_based_on_anomalies(reconstruction_errors, threshold):
    return (reconstruction_errors > threshold).astype(int)  # 1: 異常, 0: 正常

def load_node_features(standardized_data_folder, data_file):
    file_path = os.path.join(standardized_data_folder, data_file)
    principal_stress_sum = np.load(file_path)[:3654]
    x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
    y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
    z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]
    node_features = np.vstack((x_coords, y_coords, z_coords, principal_stress_sum)).T
    return node_features

def prepare_data(node_features, edge_index):
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)
    return data

# 予測結果をファイル名ごとにCSVに保存し、0以外の個数をカウントする関数
def save_predictions_by_file(predictions, data_files, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    num_vertices_per_file = 3654

    for i, data_file in enumerate(data_files):
        start_idx = i * num_vertices_per_file
        end_idx = start_idx + num_vertices_per_file
        file_predictions = predictions[start_idx:end_idx]

        # ファイル名を使用してCSVに保存
        base_filename = os.path.splitext(data_file)[0]
        output_csv_path = os.path.join(output_dir, f"{base_filename}_predicted_labels.csv")

        with open(output_csv_path, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['Predicted Label'])
            for label in file_predictions:
                writer.writerow([label])

        # 0以外のラベルの数をカウント
        non_zero_count = np.sum(file_predictions != 0)
        # print(f"{base_filename}: Number of non-zero predictions: {non_zero_count}")

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim = 4
    hidden_dim = 64
    num_classes = 2

    standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
    data_files = [f for f in os.listdir(standardized_data_folder) if f.endswith('.npy')]
    edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
    edge_index = torch.tensor(edges.T, dtype=torch.long)

    data_list = []
    for data_file in data_files:
        node_features = load_node_features(standardized_data_folder, data_file)
        data = prepare_data(node_features, edge_index)
        data_list.append(data)

    data_loader = DataLoader(data_list, batch_size=1)

    autoencoder = GraphAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=32).to(device)
    optimizer_autoencoder = torch.optim.Adam(autoencoder.parameters(), lr=0.001)

    autoencoder_epochs = 20
    autoencoder.train()
    for epoch in range(autoencoder_epochs):
        total_loss = 0
        for data in data_loader:
            data = data.to(device)
            optimizer_autoencoder.zero_grad()
            x_reconstructed, _ = autoencoder(data)
            loss = F.mse_loss(x_reconstructed, data.x)
            loss.backward()
            optimizer_autoencoder.step()
            total_loss += loss.item()
        print(f"Autoencoder Epoch {epoch + 1}/{autoencoder_epochs}, Loss: {total_loss / len(data_loader):.4f}")

    model_save_path = f"/home/nishioka/GNN/DDP_VGAE/VGAEmodels/autoencoder_model_{timestamp}.pth"
    torch.save(autoencoder.state_dict(), model_save_path)

    autoencoder.load_state_dict(torch.load(model_save_path))
    reconstruction_errors = anomaly_detection(data_loader, autoencoder, device)
    threshold = np.mean(reconstruction_errors) + np.std(reconstruction_errors)
    defect_labels = label_defects_based_on_anomalies(reconstruction_errors, threshold)

    for i, data in enumerate(data_list):
        data.y = torch.tensor(defect_labels[i], dtype=torch.long)

    train_loader = DataLoader(data_list, batch_size=1, shuffle=True)
    gnn_model = DefectPredictionGNN(input_dim=input_dim, hidden_dim=hidden_dim, num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss()

    epochs = 20
    for epoch in range(epochs):
        gnn_model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = gnn_model(batch)
            batch.y = batch.y.repeat(out.size(0))
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
        print(f"GNN Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

    gnn_model_save_path = f"/home/nishioka/GNN/DDP_VGAE/VGAEmodels/gnn_model_{timestamp}.pth"
    torch.save(gnn_model.state_dict(), gnn_model_save_path)

    gnn_model.eval()
    all_predictions = []

    with torch.no_grad():
        for batch in train_loader:
            batch = batch.to(device)
            pred = gnn_model(batch).argmax(dim=1)
            all_predictions.append(pred.cpu().numpy())
            # print("Predicted Defect Labels:", pred.cpu().numpy())

    all_predictions_flat = np.concatenate(all_predictions)
    output_dir = f"predicted_defect_labels_{timestamp}"

    save_predictions_by_file(all_predictions_flat, data_files, output_dir)
    # print(f"Predictions saved in directory: {output_dir}")

if __name__ == "__main__":
    main()


Using device: NVIDIA GeForce RTX 4090
Autoencoder Epoch 1/20, Loss: 0.0064
Autoencoder Epoch 2/20, Loss: 0.0008
Autoencoder Epoch 3/20, Loss: 0.0006
Autoencoder Epoch 4/20, Loss: 0.0005
Autoencoder Epoch 5/20, Loss: 0.0004
Autoencoder Epoch 6/20, Loss: 0.0003
Autoencoder Epoch 7/20, Loss: 0.0003
Autoencoder Epoch 8/20, Loss: 0.0003
Autoencoder Epoch 9/20, Loss: 0.0002
Autoencoder Epoch 10/20, Loss: 0.0002
Autoencoder Epoch 11/20, Loss: 0.0002
Autoencoder Epoch 12/20, Loss: 0.0002
Autoencoder Epoch 13/20, Loss: 0.0002
Autoencoder Epoch 14/20, Loss: 0.0002
Autoencoder Epoch 15/20, Loss: 0.0002
Autoencoder Epoch 16/20, Loss: 0.0002
Autoencoder Epoch 17/20, Loss: 0.0002
Autoencoder Epoch 18/20, Loss: 0.0002
Autoencoder Epoch 19/20, Loss: 0.0002
Autoencoder Epoch 20/20, Loss: 0.0002
GNN Epoch 1/20, Loss: 1.8125
GNN Epoch 2/20, Loss: 0.2980
GNN Epoch 3/20, Loss: 0.2503
GNN Epoch 4/20, Loss: 0.2127
GNN Epoch 5/20, Loss: 0.2989
GNN Epoch 6/20, Loss: 1.7133
GNN Epoch 7/20, Loss: 0.2695
GNN Epoc